In [57]:
""" Resampling Methods """

import numpy as np
import statsmodels.api as sm
from ISLP import load_data
from ISLP.models import (ModelSpec as MS,
                         summarize,
                         poly)
from sklearn.model_selection import train_test_split
from functools import partial
from sklearn.model_selection import (cross_validate, KFold, ShuffleSplit)
from sklearn.base import clone
from ISLP.models import sklearn_sm

In [58]:
""" The Validation Set Approach """

# use it for  estimate the test error rates that result from fitting various linear models on the Auto data set.
# train_test_split() to split the data into training and validation set
# 392 observation = 196 + 196 using test_size=196
# random seed = contain an element of randomness so that the result obtained can be reproduced precisely at a later time.

Auto = load_data('Auto')
Auto_train, Auto_valid = train_test_split(Auto, test_size=196, random_state=0)

In [59]:
# we can fit a linear regression using only the observations corresponding to the training set Auto_train.

hp_mm = MS(['horsepower'])
X_train = hp_mm.fit_transform(Auto_train)
y_train = Auto_train['mpg']
model = sm.OLS(y_train, X_train)
results = model.fit()

In [60]:
# calculate the validation MSE of our model.

X_valid = hp_mm.transform(Auto_valid)
y_valid = Auto_valid['mpg']
valid_pred = results.predict(X_valid)
np.mean((y_valid - valid_pred) ** 2)

np.float64(23.616617069669893)

In [61]:
# the validation MSE of the linear regression fit is 23.6%
# I can estimate the validation error for higher-degree polynomial regression.
# evalMSE() that takes a model string as well as a training and test set and return the MSE on the test set.

def evalMSE(terms, response, train, test):
    mm = MS(terms)
    X_train = mm.fit_transform(train)  # แก้ตรงนี้
    y_train = train[response]
    X_test = mm.transform(test)
    y_test = test[response]
    results = sm.OLS(y_train, X_train).fit()
    test_pred = results.predict(X_test)

    return np.mean((y_test - test_pred) ** 2)


In [62]:
# Using this function to estimate the validation MSE using linear, quadratic and cubic fits.
# enumerate() function here, which gives both the values and indices of objects as one iterates over a for loop.

MSE = np.zeros(3)
for idx, degree in enumerate(range(1, 4)):
    MSE[idx] = evalMSE([poly('horsepower', degree)],
                       'mpg',
                       Auto_train,
                       Auto_valid)
MSE


array([23.61661707, 18.76303135, 18.79694163])

In [63]:
# IF I choose a different traning/validation split the erros on the validation set.

Auto_train, Auto_valid = train_test_split(Auto, test_size=196, random_state=3)
MSE = np.zeros(3)
for idx, degree in enumerate(range(1, 4)):
    MSE[idx] = evalMSE([poly('horsepower', degree)],
                       'mpg',
                       Auto_train,
                       Auto_valid)
MSE

array([20.75540796, 16.94510676, 16.97437833])

In [64]:
""" Cross-Validation """

# The cross-validation estimate can be computed for any generalized linear model or in skelarn.
# sklearn_sm()
# statsmodel = take 2 additional optional arguments
# model_str can be used to specify a formula
# model_args shoudl be a dictionary of additional arguments used when fitting the model

hp_model = sklearn_sm(sm.OLS, MS(['horsepower']))
X, Y = Auto.drop(columns=['mpg']), Auto['mpg']
cv_results = cross_validate(hp_model, X, Y, cv=Auto.shape[0])
cv_err = np.mean(cv_results['test_score'])
cv_err

np.float64(24.231513517929226)

In [65]:
# The arguments to cross_validate() are = an object with the appropriate fit(), predict(), and score() methods, an array of feature X and a response Y.
# We begin by initializing the vector. This command may take a couple of seconds to run.

cv_error = np.zeros(5)
H = np.array(Auto['horsepower'])
M = sklearn_sm(sm.OLS)
for i, d in enumerate(range(1, 6)):
    X = np.power.outer(H, np.arange(d+1))
    M_CV = cross_validate(M, X, Y, cv=Auto.shape[0])
    cv_error[i] = np.mean(M_CV['test_score'])
    cv_error

In [66]:
# The outer() = applied to an operation that has 2 arguments such as add(), min() or power()
# It has two arrays as arguments, adn the forms a larger array where the operation is applied to each pair of elements of two arrays.

A = np.array([3, 5, 9])
B = np.array([2, 4])
np.add.outer(A, B)

array([[ 5,  7],
       [ 7,  9],
       [11, 13]])

In [67]:
# Normally it's K = n
# I can use KFold() to partition the data into K = 10 random groups.
# I will store the CV errors corresponding to the polynomial fits of degrees one to five.

cv_error = np.zeros(5)
cv = KFold(n_splits=10, shuffle=True, random_state=0) # use same splits for each degrees
for i, d in enumerate(range(1, 6)):
    X = np.power.outer(H, np.arange(d+1))
    M_CV = cross_validate(M, X, Y, cv=cv)
    cv_error[i] = np.mean(M_CV['test_score'])
    cv_error

In [68]:
# The computation time is much shorter than that of LOOCV.
# We still see little evidence that using cubic or higher-degree polynomial terms leads to a lower test error than simply using a quadratic fit.

validation = ShuffleSplit(n_splits=1, test_size=196, random_state=0)
results = cross_validate(hp_model, Auto.drop(['mpg'], axis=1),
                         Auto['mpg'],
                         cv=validation)
results['test_score']

array([23.61661707])

In [69]:
# I can estimate the vatiability in the test error by running:

validation = ShuffleSplit(n_splits=10, test_size=196, random_state=0)
results = cross_validate(hp_model, Auto.drop(['mpg'], axis=1), Auto['mpg'], cv=validation)
results['test_score'].mean(), results['test_score'].std()


(np.float64(23.80223266103417), np.float64(1.4218450941091905))

In [70]:
# This standard deviation is not a valid estimate of the sampling variability of the mean test score or the individual scores,
# since the randomly-selected training samples overlap and hence introduce correlations.


In [71]:
""" The Boostrap """

# it can be applied in alomost all sitations.
# alpha_func() = takes as input a dataframe D assumed to have a columns X and Y as well as a vector idx indicating which observations should be used to estimate alpha.
# The function the outputs the estimate for alpha based on the selected observations.

Portfolio = load_data('Portfolio')
def alpha_func(D, idx):
    cov_ = np.cov(D[['X', 'Y']].iloc[idx], rowvar=False)
    return ((cov_[1, 1] - cov_[0, 1]) / (cov_[0, 0] + cov_[1, 1] - 2 * cov_[0, 1]))

In [72]:
# this returns an estimate for alpha based on applying the minimum variance formula 
# to the observation indexed by the argument idx.

alpha_func(Portfolio, range(100))

np.float64(0.57583207459283)

In [73]:
# randomly select 100 observations.
# to constructing a new bootstrap data set and recomputing alpha based on a new data set.

rng = np.random.default_rng(0)
alpha_func(Portfolio, rng.choice(100, 100, replace=True))

np.float64(0.6074452469619002)

In [74]:
# make it simple with boot_SE() for computing the bootstrap standard error for arbitrary functions that takes only a data frame as an argument

def boot_SE(func, D, n=None, B=1000, seed=0):
    rng = np.random.default_rng(seed)
    first_, second_ = 0, 0
    n = n or D.shape[0]
    for _ in range(B):
        idx = rng.choice(D.index, n, replace=True,)
        value = func(D, idx)
        first_ += value
        second_ += value ** 2
    return np.sqrt(second_ / B - (first_ / B) ** 2)

In [75]:
# _ in the loop
# This is often sued if the value if the counter is unimportant and simply makes sure the loop is executed B times.

# Im gonna evaluate the accuracy of our estimate of alpha using B=1000 bootstrap replications.

alpha_SE = boot_SE(alpha_func, Portfolio, B=1000, seed=0)
alpha_SE

np.float64(0.09118176521277699)

In [76]:
# The bootstrap estimate for SE(alpha) is 0.0912.

In [77]:
""" Estimate the Accuracy of a Linear Regression Model """

# The bootstrap approach can be used to access the variability of the coefficient estimates and predictions from a stat learning method.
# Let's start by writing a generic function boot_OLS() for bootstrapping a regression model that takes a formula to define the corresponding regression
# use the clone() to make a copy of the formula that can be refit to the new dataframe.
# any derived features such as those defined by poly() will be re-fit on the resampled data frame.

def boot_OLS(model_matrix, response, D, idx):
    D_ = D.loc[idx]
    Y_ = D_[response]
    X_ = clone(model_matrix).fit_transform(D_)
    return sm.OLS(Y_, X_).fit().params

In [78]:
# partial() takes a function as an argument, and freezes some of its arguments, starting from the left.
# Use it to freeze the first two model-formula argument of boot_OLS()

hp_func = partial(boot_OLS, MS(['horsepower']), 'mpg')

In [80]:
# The hp_func() can now be used in order to create bootstrap estimates for the intercept and slope terms by randomly
# sampling from among the observations with replacements
# show its utility on 10 bootstrap samples.

Auto = Auto.reset_index(drop=True)
rng = np.random.default_rng(0)
np.array([hp_func(Auto, rng.choice(392, 392, replace=True)) for _ in range(10)])

array([[39.88064456, -0.1567849 ],
       [38.73298691, -0.14699495],
       [38.31734657, -0.14442683],
       [39.91446826, -0.15782234],
       [39.43349349, -0.15072702],
       [40.36629857, -0.15912217],
       [39.62334517, -0.15449117],
       [39.0580588 , -0.14952908],
       [38.66688437, -0.14521037],
       [39.64280792, -0.15555698]])

In [81]:
# we use the boot_SE() to compute the standard errors of 1,000 bootstrap estimates for the intercept and slope terms

hp_se = boot_SE(hp_func, Auto, B=1000, seed=10)
hp_se

intercept     0.848807
horsepower    0.007352
dtype: float64

In [82]:
# the bootstrap estimate for SE(Beta0) is 0.85.
# the boothstrap estimate for SE(Beta1) is 0.0074
# Standard formulas can be used to compute the standard erros for the ression coeeficients in the linear model!

hp_model.fit(Auto, Auto['mpg'])
model_se = summarize(hp_model.results_)['std err']
model_se

intercept     0.717
horsepower    0.006
Name: std err, dtype: float64

In [83]:
# It suggest the opporsite.
# Let's compute the bootstrap standard error estimates and the standard linear regression estimates that result from fitting the quadratic model to the data.

quad_model = MS([poly('horsepower', 2, raw=True)])
quad_func = partial(boot_OLS, quad_model, 'mpg')
boot_SE(quad_func, Auto, B=1000)

intercept                                  2.067840
poly(horsepower, degree=2, raw=True)[0]    0.033019
poly(horsepower, degree=2, raw=True)[1]    0.000120
dtype: float64

In [84]:
# we compare the results to the standard errors computed using sm.OLS()

M = sm.OLS(Auto['mpg'], quad_model.fit_transform(Auto))
summarize(M.fit())['std err']

intercept                                  1.800
poly(horsepower, degree=2, raw=True)[0]    0.031
poly(horsepower, degree=2, raw=True)[1]    0.000
Name: std err, dtype: float64